# Classifying a ticket using LLM (Claude)

In [7]:
import json
import re
import anthropic
import os
from dotenv import load_dotenv
load_dotenv()

from datetime import datetime
import time

In [5]:
ANTHRPOIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key = ANTHRPOIC_API_KEY)
model = "claude-sonnet-5-5"

In [4]:
def _parse_json(text: str) -> dict:
    text = text.strip()
    m = re.search(r'```(?:json)?\s*\n?([\s\S]+?)\n?\s*```', text)
    if m:
        text = m.group(1).strip()
    start, end = text.find('{'), text.find('}')+1
    if start != -1 and end > start:
        text = text[start:end]
    return json.loads(text)

In [9]:
CLASSIFY_SYSTEM = """
You are a support ticket classifier for Meridian cloud.

Classify each ticket into exactly one of these categories:
    - billing_dispute: customer contests a charge or invoice total
    - billing_inquiry: customer asks about charges without disputing them
    - payment_failure: a payment attempt was declined or could not be processed
    - technical_error: a product or API is not working as expected
    - account_access: login, password, or permission issues
    - feature_request: customer asks for capability that doesn't exist yet
    - other: doesn't fit the above categories

Return a JSON object with fields "category" and "confidence" (high/medium/low).

Examples:

Input: "I was charged $120 but my plan is $80/month. Please explain.
Output: {"category": "billing_dispute", "confidence": "high"}

Input: "When does my billing cycle reset every month?"
Output: {"category": "billing_inquiry", "confidence": "high"}

Input: "My card ending with 1234 was declined when I tried to upgrade."
Output: {"category": "payment_failure", "confidence": "high"}

Input: "The /v2/events endpoint has been returning 503 error since morning."
Output: {"category": "technical_error", "confidence": "high"}

Input: "I need to add a team member but the invite button is disabled."
Output: {"category": "account_access", "confidence": "medium"}

Return only a JSON object.
"""

In [11]:
def classify_ticket(message: str) -> dict:
    response = client.messages.create(
        model = model,
        max_tokens = 128,
        system = CLASSIFY_SYSTEM,
        messages = [{"role": "user", "content": message}]
    )
    for block in response.content:
        if block.type == "text":
            return _parse_json(block.text)
    raise ValueError("No text block in response.")

tickets = [
        "I was charged twice this month — two identical $99 charges on the 1st and 2nd.",
        "How do I export my invoice as a PDF?",
        "Our webhook endpoint stopped receiving events at 14:00 UTC today.",
        "Can you add bulk CSV import for contacts? We need it for our migration.",
        "I can't log in — it says my account is suspended but I received no email about it.",
    ]

start_time = time.perf_counter_ns()
for ticket in tickets:
        result = classify_ticket(ticket)
        print(f"Ticket     : {ticket[:70]}{'...' if len(ticket) > 70 else ''}")
        print(f"Category   : {result.get('category')}")
        print(f"Confidence : {result.get('confidence')}")
        print()

elapsed_ms = (time.perf_counter_ns() - start_time) / 1_000_000
print(f"Duration: {elapsed_ms:.2f} ms")


Ticket     : I was charged twice this month — two identical $99 charges on the 1st ...
Category   : billing_dispute
Confidence : high

Ticket     : How do I export my invoice as a PDF?
Category   : billing_inquiry
Confidence : medium

Ticket     : Our webhook endpoint stopped receiving events at 14:00 UTC today.
Category   : technical_error
Confidence : high

Ticket     : Can you add bulk CSV import for contacts? We need it for our migration...
Category   : feature_request
Confidence : high

Ticket     : I can't log in — it says my account is suspended but I received no ema...
Category   : account_access
Confidence : high

Duration: 8619.75 ms
